# AI-Lake — Agent Memory (Phase 9)

Demonstrates **agent-scoped partition isolation**, **`ToolCallSchema`**, **`EpisodicMemorySchema`**,
**`ScoreFn` hybrid scoring**, and the **`ailake.Agent`** high-level helper.

| Section | Topic |
|---|---|
| 1 | `ailake.Agent` — remember, recall, log_tool_call, assemble_context |
| 2 | Partition isolation — `partition_by`, `partition_value`, `partition_filter` |
| 3 | `ToolCallSchema` — logging and semantic search over tool history |
| 4 | `EpisodicMemorySchema` — recency decay + importance scoring |
| 5 | `ScoreFn` — hybrid ranking (distance × recency × importance) |
| 6 | `assemble_context()` for agent memory |
| 7 | Pre-generated agent fixture (`init_demo.py`) |

> **Pre-requisite**: `01_ailake_demo.ipynb §24–28` covers the same features as a quick tour.
> This notebook goes deeper with full examples and larger fixtures.

## 0. Setup

In [ ]:
import datetime, json, math, os, pathlib, random

import numpy as np
import pandas as pd
import pyarrow as pa

import ailake
from ailake import TableWriter

AGENT_PATH = os.environ.get('DEMO_AGENT_PATH', '/data/ailake_agent')
TABLE_PATH = os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')
BASE       = pathlib.Path(AGENT_PATH).parent

with open(BASE / 'demo_query.json') as f:
    demo_meta = json.load(f)

DIM    = int(demo_meta['dim'])
METRIC = demo_meta['metric']

print(f'AGENT_PATH : {AGENT_PATH}')
print(f'DIM        : {DIM}')
print(f'METRIC     : {METRIC}')

In [ ]:
def synthetic_embed(texts: list) -> list:
    """Deterministic unit-vector embed — replace with real model in production."""
    out = []
    for t in texts:
        rng = random.Random(hash(t) & 0xFFFFFFFF)
        v   = [rng.gauss(0, 1) for _ in range(DIM)]
        nm  = math.sqrt(sum(x * x for x in v)) or 1.0
        out.append([x / nm for x in v])
    return out

# Verify embed function works
v = synthetic_embed(['test'])[0]
print(f'embed dim={len(v)}  norm={math.sqrt(sum(x*x for x in v)):.6f}  (expected 1.0)')

## 1. `ailake.Agent` — high-level agent memory helper

`ailake.Agent(table_path, embed_fn, agent_id)` wraps `TableWriter` + `search` +
`assemble_context` with an agent-scoped partition.

| Method | Description |
|---|---|
| `remember(text, importance=1.0)` | Embed + persist a memory chunk; returns snapshot_id |
| `recall(query, top_k=5)` | Vector search scoped to this agent's partition |
| `log_tool_call(name, input, output, outcome, latency_ms)` | Store a tool call record |
| `assemble_context(query, max_tokens)` | Ranked context XML for this agent's memories |

All methods respect `partition_by='agent_id'` automatically — no cross-agent leakage.

In [ ]:
NOTEBOOK_AGENT_PATH = str(BASE / 'nb_agent_helper')

# Two agents sharing the same table — fully isolated by partition
agent_x = ailake.Agent(NOTEBOOK_AGENT_PATH, embed_fn=synthetic_embed, agent_id='agent-X')
agent_y = ailake.Agent(NOTEBOOK_AGENT_PATH, embed_fn=synthetic_embed, agent_id='agent-Y')

print(f'agent_x: {agent_x}')
print(f'agent_y: {agent_y}')

In [ ]:
# agent-X remembers facts about vector databases
memories_x = [
    ('AI-Lake stores embeddings and tabular data in a single Parquet file.', 0.9),
    ('HNSW provides approximate nearest neighbour search in sub-linear time.', 0.8),
    ('Product quantization reduces storage from 6 KB to 48 bytes per vector.', 0.7),
    ('Iceberg time-travel allows rollback to any historical snapshot.', 0.6),
    ('Geometric pruning eliminates 95-99% of files before any I/O.', 0.85),
]
for text, importance in memories_x:
    mem_id = agent_x.remember(text, importance=importance)
    print(f'  [X] remembered: "{text[:55]}..."  mem_id={mem_id[:8]}')

print()

# agent-Y remembers facts about LLM inference
memories_y = [
    ('RAG pipelines retrieve context chunks before generating a response.', 0.9),
    ('Context window limits require careful chunk size selection.', 0.75),
    ('Reranking improves precision by applying a cross-encoder after retrieval.', 0.8),
]
for text, importance in memories_y:
    mem_id = agent_y.remember(text, importance=importance)
    print(f'  [Y] remembered: "{text[:55]}..."  mem_id={mem_id[:8]}')

# commit() persists buffered memories to disk
snap_x = agent_x.commit()
snap_y = agent_y.commit()
print(f'\nCommitted: agent-X snap={snap_x}  agent-Y snap={snap_y}')

In [ ]:
# recall() is partition-scoped — agent-X only sees its own memories
results_x = agent_x.recall('vector search index performance', top_k=3)
print('agent-X recall for "vector search index performance":')
for r in results_x:
    print(f'  dist={r["distance"]:.4f}  {r.get("chunk_text", r.get("text", ""))[:70]}')

print()

# agent-Y gets different results — no cross-contamination
results_y = agent_y.recall('vector search index performance', top_k=3)
print('agent-Y recall for same query (no agent-X memories visible):')
for r in results_y:
    print(f'  dist={r["distance"]:.4f}  {r.get("chunk_text", r.get("text", ""))[:70]}')

## 2. Partition isolation — `partition_by` / `partition_filter`

Partition isolation is a two-part feature:

**Write side** — `TableWriter(partition_by='agent_id', partition_value='agent-X')` creates
an Iceberg identity partition. Every `write_batch()` call writes rows tagged with that value.

**Read side** — `ailake.search(..., partition_filter='agent-X')` prunes the Iceberg
manifest before the centroid geometry check. Files for other partitions are **never opened**.

Performance: with 1000 agents and 100 files each, a query for `agent-X` touches
100 files (its own) instead of 100 000 (the whole table).

In [ ]:
# Read directly via ailake.search — demonstrate partition_filter
q = np.array(synthetic_embed(['HNSW graph construction'])[0], dtype=np.float32)

# No filter: both partitions searched
all_r  = ailake.search(NOTEBOOK_AGENT_PATH, q, top_k=20).to_pandas()
# agent-X only
x_only = ailake.search(NOTEBOOK_AGENT_PATH, q, top_k=20, partition_filter='agent-X').to_pandas()
# agent-Y only
y_only = ailake.search(NOTEBOOK_AGENT_PATH, q, top_k=20, partition_filter='agent-Y').to_pandas()

print(f'No filter  : {len(all_r):3d} rows')
print(f'agent-X    : {len(x_only):3d} rows')
print(f'agent-Y    : {len(y_only):3d} rows')

# Isolation check: files (not row_id) must never overlap across partitions.
# row_id is per-file sequential (starts at 0 in each shard) so overlap is expected.
x_files = set(x_only['file'].tolist())
y_files = set(y_only['file'].tolist())
file_overlap = x_files & y_files
print(f'File overlap: {len(file_overlap)} (expected 0)')
assert len(file_overlap) == 0, f'Partition leak: shared files = {file_overlap}'
print('PASS — partitions are fully isolated')

In [ ]:
# Pre-generated fixture: agent-A and agent-B from init_demo.py
fixture_meta = demo_meta.get('agent', {})
print('Fixture agent metadata:')
print(json.dumps(fixture_meta, indent=2))

q_fixture = np.array(demo_meta['query_vector'], dtype=np.float32)

a_fix = ailake.search(AGENT_PATH, q_fixture, top_k=5, partition_filter='agent-A').to_pandas()
b_fix = ailake.search(AGENT_PATH, q_fixture, top_k=5, partition_filter='agent-B').to_pandas()
print(f'\nagent-A fixture: {len(a_fix)} results')
print(f'agent-B fixture: {len(b_fix)} results')

## 3. `ToolCallSchema` — logging and semantic search over tool history

`ToolCallSchema` extends `LlmContextSchema` with agent execution metadata:

| Column | Type | Description |
|---|---|---|
| `agent_id` | string | Agent identifier |
| `session_id` | string | Conversation / run session |
| `step_index` | uint32 | Step number within the session |
| `tool_name` | string | Tool that was called |
| `tool_input_json` | string | JSON-serialised input arguments |
| `tool_output_json` | string | JSON-serialised output |
| `outcome` | string | `"success"` / `"failure"` / `"timeout"` |
| `latency_ms` | uint32 | Wall-clock latency in milliseconds |
| `chunk_text` | string | `"tool_name: input | output"` — the embedded text |
| `embedding` | Vector | Semantic vector of the call context |

**Why embed tool calls?** Lets you ask: *"Have I done something similar before?",
"When did web_search fail on queries like this?"*, enabling reflexive agent behaviour.

In [ ]:
TOOL_CALL_PATH = str(BASE / 'nb_agent_tool_calls')

# Simulate two sessions, each with several tool calls
sessions = {
    'session-100': [
        ('web_search', {'q': 'HNSW vs IVF-PQ'},   {'snippets': ['HNSW better recall, IVF-PQ better storage.']}, 'success', 210),
        ('web_search', {'q': 'DuckDB HNSW support'}, {'snippets': ['DuckDB has a VSS extension for HNSW.']}, 'success', 195),
        ('code_exec',  {'lang': 'python', 'code': 'import ailake; print(ailake.__version__)'},
                       {'stdout': '0.0.20'},  'success',  8),
        ('web_search', {'q': 'pgvector limitations'}, {'error': 'rate_limited'}, 'failure', 5001),
    ],
    'session-101': [
        ('web_search', {'q': 'Apache Iceberg vs Delta Lake'}, {'snippets': ['Iceberg has better partition evolution.']}, 'success', 320),
        ('code_exec',  {'lang': 'sql', 'code': 'SELECT count(*) FROM ailake.demo'},
                       {'rows': [{'count': 500}]},  'success', 45),
        ('web_search', {'q': 'vector quantization product quantization'},
                       {'snippets': ['PQ encodes sub-vectors independently.']}, 'success', 280),
    ],
}

all_rows   = []
chunk_texts = []
for session_id, calls in sessions.items():
    for step, (tool_name, inp, out, outcome, latency) in enumerate(calls):
        ct = f'{tool_name}: {json.dumps(inp)} | {json.dumps(out)}'
        all_rows.append({
            'chunk_text'      : ct,
            'agent_id'        : 'agent-X',
            'session_id'      : session_id,
            'step_index'      : step,
            'tool_name'       : tool_name,
            'tool_input_json' : json.dumps(inp),
            'tool_output_json': json.dumps(out),
            'outcome'         : outcome,
            'latency_ms'      : latency,
        })
        chunk_texts.append(ct)

tc_embs = np.array(synthetic_embed(chunk_texts), dtype=np.float32)

w = TableWriter(TOOL_CALL_PATH, dim=DIM, metric=METRIC,
                partition_by='agent_id', partition_value='agent-X')
w.write_batch(
    chunk_texts,
    tc_embs.tolist(),
    extra_columns={k: [r[k] for r in all_rows] for k in all_rows[0] if k != 'chunk_text'},
)
w.commit()

print(f'Logged {len(all_rows)} tool calls across {len(sessions)} sessions.')

In [ ]:
# Semantic search over tool call history
# chunk_texts are stored in the 'text' column by write_batch()
def search_tool_calls(query: str, top_k: int = 5) -> pd.DataFrame:
    q = np.array(synthetic_embed([query])[0], dtype=np.float32)
    return ailake.search(
        TOOL_CALL_PATH, q,
        top_k=top_k,
        fetch_data=True,
        partition_filter='agent-X',
    ).to_pandas()[['tool_name', 'outcome', 'latency_ms', 'text', '_distance']]

print('=== Query: "vector search and indexing" ===')
print(search_tool_calls('vector search and indexing', top_k=3).to_string(index=False))

print()
print('=== Query: "failed or timed out tool calls" ===')
print(search_tool_calls('failed or timed out tool calls', top_k=3).to_string(index=False))

In [ ]:
# Filter by outcome post-search
all_calls = ailake.search(
    TOOL_CALL_PATH,
    np.array(synthetic_embed(['any tool call'])[0], dtype=np.float32),
    top_k=50,
    fetch_data=True,
    partition_filter='agent-X',
).to_pandas()

summary = all_calls.groupby(['tool_name', 'outcome']).size().reset_index(name='count')
print('Tool call summary by outcome:')
print(summary.to_string(index=False))

## 4. `EpisodicMemorySchema` — recency decay + importance scoring

`EpisodicMemorySchema` adds recency and importance signals to long-term memory:

| Column | Type | Description |
|---|---|---|
| `recency_weight` | float32 | `exp(-λ × days_since_access)` — 1.0 = fresh, → 0 = stale |
| `access_count` | uint32 | How many times this memory was recalled |
| `importance_score` | float32 | Agent-assigned relevance (0–1) |
| `last_accessed_at` | string (ISO 8601) | Timestamp of last access |

**Decay curve**: with `λ = 0.05`, a 14-day-old memory has `recency_weight ≈ 0.50`;
a 60-day-old memory has `recency_weight ≈ 0.05`.

**`MemoryDecayJob`** (Phase 9, pending) recomputes `recency_weight` for all rows
periodically via compaction — no need to touch individual files.

In [ ]:
EPISODIC_PATH = str(BASE / 'nb_agent_episodic')
now           = datetime.datetime.utcnow()
LAMBDA        = 0.05  # daily decay constant

# 15 memories with varied ages and importance
memory_specs = [
    # (text, days_old, access_count, importance)
    ('HNSW graph M=16 gives good recall for RAG workloads.',      0,  8, 0.95),
    ('PQ codebook trained on first 10k rows generalises well.',   1,  3, 0.80),
    ('Iceberg snapshot V1 contained malformed AVRO manifest.',    2,  0, 0.40),
    ('Context window of 128k tokens fits ~300 chunks of 400t.',   3,  5, 0.85),
    ('DuckDB ailake_search() extension loaded correctly.',        5,  2, 0.60),
    ('Trino VectorScanConnector requires plugin on all workers.', 7,  1, 0.70),
    ('Geometric pruning eliminated 97% of files in bench.',      10,  4, 0.90),
    ('S3 multipart upload needed for files > 100 MB.',           14,  1, 0.55),
    ('Residual-PQ recall@10 = 94.2% vs 91.8% for plain PQ.',    20,  0, 0.75),
    ('Flink VectorScanSource saw 40k events/s throughput.',       28,  2, 0.65),
    ('Go client SearchOptions.PartitionFilter added in v0.0.18.', 35, 0, 0.50),
    ('BinaryHamming removed — recall ≈ 0 on float embeddings.',   45, 0, 0.30),
    ('IVF-PQ k-means++ init now O(n×k) instead of O(n×k²).',     60, 1, 0.70),
    ('create_or_open part_counter bug fixed in v0.0.17.',         90, 0, 0.45),
    ('mmap HNSW loading reduced cold-start from 3s to 120ms.',  180, 3, 0.80),
]

ep_texts   = [m[0] for m in memory_specs]
ep_embs    = np.array(synthetic_embed(ep_texts), dtype=np.float32)
rw         = [math.exp(-LAMBDA * m[1]) for m in memory_specs]
ac         = [m[2] for m in memory_specs]
im         = [m[3] for m in memory_specs]
accessed_at = [(now - datetime.timedelta(days=m[1])).isoformat() for m in memory_specs]

w = TableWriter(EPISODIC_PATH, dim=DIM, metric=METRIC,
                partition_by='agent_id', partition_value='agent-X')
w.write_batch(
    ep_texts, ep_embs.tolist(),
    extra_columns={
        'agent_id'        : ['agent-X'] * len(memory_specs),
        'recency_weight'  : rw,
        'access_count'    : ac,
        'importance_score': im,
        'last_accessed_at': accessed_at,
    },
)
w.commit()

df_ep = pd.DataFrame({
    'text'            : [t[:55] for t in ep_texts],
    'days_old'        : [m[1] for m in memory_specs],
    'recency_weight'  : [f'{r:.3f}' for r in rw],
    'importance'      : im,
})
print('Episodic memories written:')
print(df_ep.to_string(index=False))

## 5. `ScoreFn` — hybrid ranking (distance × recency × importance)

Default ranking uses pure vector distance. `score_fn` injects a custom formula:

```python
ScoreFn = Callable[[distance: float, row: dict], float]
```

**Convention**: lower score = ranked first (same as distance).  
**Hybrid formula**: `score = distance / (recency_weight × importance_score)`

This boosts fresh + important memories: a memory with `recency=1.0, importance=0.9`
scores `distance / 0.9` — close to pure distance. A stale memory with `recency=0.05,
importance=0.3` scores `distance / 0.015` — strongly downranked.

In [ ]:
def episodic_score(distance: float, row: dict) -> float:
    rw = row.get('recency_weight',   1.0)
    im = row.get('importance_score', 1.0)
    return distance / max(rw * im, 1e-9)

query = np.array(synthetic_embed(['performance and recall improvements'])[0], dtype=np.float32)

pure_df = ailake.search(
    EPISODIC_PATH, query, top_k=8,
    fetch_data=True, partition_filter='agent-X',
).to_pandas()[['_distance', 'recency_weight', 'importance_score']].assign(
    text=[m[0][:50] for m in memory_specs[:8]],
    rank_pure=range(1, 9),
)

hybrid_df = ailake.search(
    EPISODIC_PATH, query, top_k=8,
    fetch_data=True, partition_filter='agent-X',
    score_fn=episodic_score,
).to_pandas()[['_distance', 'recency_weight', 'importance_score']].assign(
    rank_hybrid=range(1, 9),
)

print('Pure distance ranking (top-8):')
print(pure_df[['rank_pure', '_distance', 'recency_weight', 'importance_score']].to_string(index=False))
print()
print('Hybrid ranking (top-8) — stale/low-importance memories downranked:')
print(hybrid_df[['rank_hybrid', '_distance', 'recency_weight', 'importance_score']].to_string(index=False))

In [ ]:
# Visualise recency decay curve for different λ values
print('Recency weight by days old (decay formula: exp(-λ × days))\n')
print(f'  {"days":>5}   λ=0.01   λ=0.05   λ=0.10   λ=0.20')
print(f'  {"-"*5}   {("-"*7+"  ")*4}')
for days in [0, 1, 7, 14, 30, 60, 90, 180, 365]:
    vals = [math.exp(-lam * days) for lam in [0.01, 0.05, 0.10, 0.20]]
    print(f'  {days:>5}   {"   ".join(f"{v:.4f}" for v in vals)}')

## 6. `assemble_context()` for agent memory

`ailake.assemble_context()` accepts any list of dicts with at least `chunk_text`,
`document_id`, `chunk_index`, and `distance`. It deduplicates near-identical chunks
and renders structured XML within a token budget — ready for Claude/GPT-4/Gemini.

For agent memory, use `Agent.assemble_context(query, max_tokens)` which combines
`recall()` + `assemble_context()` in a single call.

In [ ]:
import shutil

# Use Agent.assemble_context() — wraps recall + context assembly
AGENT_CTX_PATH = str(BASE / 'nb_agent_ctx')
shutil.rmtree(AGENT_CTX_PATH, ignore_errors=True)
agent_ctx = ailake.Agent(AGENT_CTX_PATH, embed_fn=synthetic_embed, agent_id='agent-ctx')

# Populate with memories
ctx_memories = [
    'AI-Lake partitions by agent_id for full isolation between agents.',
    'HNSW M=16 ef_construction=150 is the default for general RAG workloads.',
    'Geometric pruning checks centroid distance before downloading Parquet files.',
    'EpisodicMemorySchema adds recency_weight that decays exponentially with age.',
    'ScoreFn lets agents inject custom ranking without rewriting the index.',
    'ToolCallSchema embeds tool name + input + output for semantic search over history.',
    'assemble_context() respects a token budget and deduplicates similar chunks.',
]
for text in ctx_memories:
    agent_ctx.remember(text, importance=0.8)

# commit() must be called before recall/assemble_context
agent_ctx.commit()

# assemble_context() returns XML ready to inject into a system prompt
ctx_xml = agent_ctx.assemble_context('how does AI-Lake handle agent isolation?', max_tokens=1024)
print(ctx_xml)

In [ ]:
# Manual path: recall() + assemble_context() for more control
query = 'scoring and ranking of retrieved memories'
recalled = agent_ctx.recall(query, top_k=5)

# Shape for assemble_context
chunks = [
    {
        'document_id'  : f'mem-{i:03d}',
        'chunk_index'  : 0,
        'chunk_text'   : r.get('chunk_text', r.get('text', '')),
        'document_title': 'Agent Memory',
        'distance'     : float(r['distance']),
    }
    for i, r in enumerate(recalled)
]

xml = ailake.assemble_context(chunks=chunks, max_tokens=512, dedup_threshold=0.05)
print(xml)

## 7. Pre-generated agent fixture (`init_demo.py`)

`init_demo.py` creates two pre-populated agent partitions at container startup:

| Agent | Rows | Topics |
|---|---|---|
| `agent-A` | 50 | Topics 0–49 of TOPICS corpus |
| `agent-B` | 50 | Topics 50–99 of TOPICS corpus |

Both partitions live in `/data/ailake_agent` (env: `DEMO_AGENT_PATH`).
This section verifies the fixture and shows a cross-agent comparison.

In [ ]:
import pyarrow.parquet as pq

parquet_files = sorted(pathlib.Path(AGENT_PATH).rglob('*.parquet'))
print(f'Parquet files in fixture: {len(parquet_files)}')
for p in parquet_files:
    tbl = pq.read_table(str(p))
    print(f'  {p.name}  rows={tbl.num_rows}')

print()
q_fix = np.array(demo_meta['query_vector'], dtype=np.float32)

for agent_id in ['agent-A', 'agent-B']:
    r = ailake.search(AGENT_PATH, q_fix, top_k=3,
                      partition_filter=agent_id, fetch_data=True).to_pandas()
    print(f'{agent_id} top-3 results:')
    for _, row in r.iterrows():
        preview = row.get('text', row.get('chunk_text', ''))[:65]
        print(f'  dist={row["_distance"]:.4f}  "{preview}"')
    print()

## Next Steps

| Resource | Location |
|---|---|
| Phase 9 spec | `docs/specs/LLM_CONTEXT.md §Phase 9` |
| Python API — Agent | `ailake-py/README.md §Agent` |
| DuckDB partition_filter | `duckdb-ailake/README.md` |
| JVM partition fields | `docs/specs/JVM_PLUGINS.md §Phase 9` |
| Full SDK walkthrough | `01_ailake_demo.ipynb §24–28` |
| Multimodal deep-dive | `07_multimodal.ipynb` |